# KKBOX Churn Prediction — Base Preprocessing Notebook

이 노트북은 **모델링 이전의 공통 전처리 과정**만 포함합니다.

포함 범위:
- 데이터 로드
- Target 확인
- 기본 변수 정리
- Train / Test Split
- 결측치 처리 및 불필요 변수 제거
- Feature Engineering
- 파생변수 생성 결과 점검

> 이 노트북 이후부터 ML / DL 모델별 전처리 및 모델링을 별도로 진행합니다.


## 1. Library & Data Load


In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 내 PC에서 사용하는 실제 데이터 경로
DATA_PATH = Path(
    r"/content/drive/MyDrive/colab tutorial/2차단위/머신러닝 모델링/data/integrated_data_final.csv"
)

# 다른 팀원이 저장소를 clone한 경우를 위한 상대경로 fallback
if not DATA_PATH.exists():
    DATA_PATH = Path("data/processed/integrated_data.csv")

print("Data path:", DATA_PATH.resolve())

df = pd.read_csv(DATA_PATH)

print("shape:", df.shape)
display(df.head())


## 2. Target 확인

In [ ]:
print("Target count")
display(df["is_churn"].value_counts().sort_index().to_frame())

print("Target ratio")
display(df["is_churn"].value_counts(normalize=True).sort_index().to_frame())


## 3. Split 이전 기본 정리

기존 모델 비교 노트북의 기준을 그대로 사용합니다.

- `bd`: 10~80세만 정상값으로 처리
- 정상 범위를 벗어난 나이는 결측 처리 후 `age_group='unknown'`
- `gender` 결측치는 `Unknown`
- `registration_init_time`은 날짜형으로 변환하되 모델 입력에서는 제외


In [ ]:
df["registration_init_time"] = pd.to_datetime(
    df["registration_init_time"],
    format="%Y%m%d",
    errors="coerce"
)

df.loc[~df["bd"].between(10, 80), "bd"] = np.nan

df["age_group"] = pd.cut(
    df["bd"],
    bins=[9, 19, 29, 39, 49, 59, 69, 80],
    labels=["10s", "20s", "30s", "40s", "50s", "60s", "70_80"]
)

df["age_group"] = (
    df["age_group"]
    .cat.add_categories("unknown")
    .fillna("unknown")
)

df["gender"] = df["gender"].fillna("Unknown")


## 4. Train / Test Split

In [ ]:
X = df.drop(columns="is_churn")
y = df["is_churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train = X_train.copy()
X_test = X_test.copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print(f"Train churn ratio: {y_train.mean():.4f}")
print(f"Test churn ratio : {y_test.mean():.4f}")


## 5. 결측치 처리 및 불필요 변수 제거

Train 데이터에서 계산한 값만 이용해 Test 데이터를 처리하여 데이터 누수를 방지합니다.

- `city`: 제외
- `registered_via`: Train 최빈값
- `last_auto_renew`, `last_is_cancel`, `last_plan_days`: Train 최빈값
- `days_since_last_log`: Train 평균
- `registration_init_time`, `cancel_on_last_date`, `days_to_expire`, `bd`: 모델 입력에서 제외


In [ ]:
X_train = X_train.drop(columns=["city"])
X_test = X_test.drop(columns=["city"])

registered_via_mode = X_train["registered_via"].mode()[0]
X_train["registered_via"] = X_train["registered_via"].fillna(registered_via_mode)
X_test["registered_via"] = X_test["registered_via"].fillna(registered_via_mode)

for col in ["last_auto_renew", "last_is_cancel", "last_plan_days"]:
    fill_value = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(fill_value)
    X_test[col] = X_test[col].fillna(fill_value)

last_log_mean = X_train["days_since_last_log"].mean()
X_train["days_since_last_log"] = X_train["days_since_last_log"].fillna(last_log_mean)
X_test["days_since_last_log"] = X_test["days_since_last_log"].fillna(last_log_mean)

drop_cols = [
    "registration_init_time",
    "cancel_on_last_date",
    "days_to_expire",
    "bd"
]

X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

print("Remaining nulls before feature engineering")
display(
    pd.DataFrame({
        "train_null": X_train.isna().sum(),
        "test_null": X_test.isna().sum()
    }).query("train_null > 0 or test_null > 0")
)


## 6. Feature Engineering

나눗셈으로 만드는 파생변수는 **safe division**을 사용합니다.

- 분모가 0이면 해당 파생값을 `0`으로 처리
- `inf`, `-inf`가 생성되는 것을 방지
- 이후 DL 전처리 직전 숫자형 변수의 남은 `NaN / inf`도 한 번 더 정리


In [ ]:
def safe_divide(numerator, denominator):
    """분모가 0이면 0을 반환하는 안전한 나눗셈 함수"""
    num = numerator.to_numpy(dtype=float)
    den = denominator.to_numpy(dtype=float)

    result = np.zeros(len(numerator), dtype=float)
    np.divide(
        num,
        den,
        out=result,
        where=(den != 0)
    )

    return pd.Series(result, index=numerator.index)


def add_features(data):
    data = data.copy()

    # 0 값 자체가 의미를 가질 수 있는 보조 flag
    data["no_unique_song"] = (data["total_num_unq"] == 0).astype(int)
    data["zero_plan_days"] = (data["last_plan_days"] == 0).astype(int)

    # 거래 관련
    data["cancel_rate"] = safe_divide(
        data["cancel_count"],
        data["transaction_count"]
    )

    data["avg_payment"] = safe_divide(
        data["total_payment"],
        data["transaction_count"]
    )

    # 이용 로그 관련
    data["avg_daily_secs"] = safe_divide(
        data["total_secs"],
        data["activity_days"]
    )

    data["avg_daily_complete"] = safe_divide(
        data["total_num_100"],
        data["activity_days"]
    )

    data["avg_daily_unq"] = safe_divide(
        data["total_num_unq"],
        data["activity_days"]
    )

    data["complete_per_unq"] = safe_divide(
        data["total_num_100"],
        data["total_num_unq"]
    )

    # 결제 효율 관련
    data["payment_per_plan_day"] = safe_divide(
        data["avg_payment"],
        data["last_plan_days"]
    )

    return data


X_train = add_features(X_train)
X_test = add_features(X_test)


In [ ]:
new_features = [
    "cancel_rate",
    "avg_payment",
    "avg_daily_secs",
    "avg_daily_complete",
    "avg_daily_unq",
    "complete_per_unq",
    "payment_per_plan_day"
]

display(X_train[new_features].describe().T)

print("Remaining NaN after safe division")
display(
    pd.DataFrame({
        "train_null": X_train[new_features].isna().sum(),
        "test_null": X_test[new_features].isna().sum()
    })
)

numeric_cols = X_train.select_dtypes(include=np.number).columns

print("Train inf count:", np.isinf(X_train[numeric_cols]).sum().sum())
print("Test inf count :", np.isinf(X_test[numeric_cols]).sum().sum())


In [ ]:
X_train.columns

## 7. Base Preprocessing 완료

여기까지가 모든 모델에서 공통으로 사용할 **base preprocessing + feature engineering** 단계입니다.

현재 준비된 객체:
- `X_train`, `X_test`
- `y_train`, `y_test`

다음 단계부터는 모델별로 분기합니다.

예를 들어 DL 모델링에서는 이후에 다음 작업을 직접 진행할 수 있습니다.

1. `msno` 제거
2. 범주형 변수 인코딩
3. 수치형 변수 Scaling
4. Train / Validation 분리
5. Tensor / DataLoader 생성
6. MLP 모델 정의 및 학습



In [ ]:
# msno는 단순 식별자이므로 drop하고 MLP용 데이터셋 생성
X_train_mlp = X_train.drop(columns=["msno"]).copy()
X_test_mlp = X_test.drop(columns=["msno"]).copy()

# PyTorch MLP는 NaN / inf를 직접 처리하지 못하므로 숫자형 변수만 최종 정리
numeric_mlp_cols = X_train_mlp.select_dtypes(include=np.number).columns

X_train_mlp[numeric_mlp_cols] = (
    X_train_mlp[numeric_mlp_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

X_test_mlp[numeric_mlp_cols] = (
    X_test_mlp[numeric_mlp_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("MLP train numeric NaN:", X_train_mlp[numeric_mlp_cols].isna().sum().sum())
print("MLP test numeric NaN :", X_test_mlp[numeric_mlp_cols].isna().sum().sum())
print("MLP train numeric inf:", np.isinf(X_train_mlp[numeric_mlp_cols]).sum().sum())
print("MLP test numeric inf :", np.isinf(X_test_mlp[numeric_mlp_cols]).sum().sum())


In [ ]:
# train, validation 분리
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_mlp,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [ ]:
# 범주형 변수 인코딩 - One Hot Encoding
X_train_mlp.columns

In [ ]:
# cat col: gender, registered_via, age_group
# last_auto_renew, last_is_cancel, has_transaction, has_log, zero_plan_days

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = [
    "gender",
    "registered_via",
    "age_group"
]

binary_cols = [
    'last_auto_renew',
    'last_is_cancel',
    'has_transaction',
    'has_log',
    'zero_plan_days'
]

num_cols = [
    col for col in X_train_mlp.columns
    if col not in cat_cols + binary_cols
]



In [ ]:
# One Hot Encoding + Scaling

# 수치형 변수: StandardScaler
# 범주형 변수: OneHotEncoder
# 0/1 이진 변수: 그대로 통과(passthrough)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("binary", "passthrough", binary_cols)
    ]
)


In [ ]:
# 전처리 진행

X_tr = preprocessor.fit_transform(X_tr)
X_val = preprocessor.transform(X_val)
X_test_mlp = preprocessor.transform(X_test_mlp)

In [ ]:
type(X_tr)

In [ ]:
# 이제 딥러닝 모델링을 위해 torch로 변환
import torch

X_tr_tensor = torch.tensor(X_tr, dtype = torch.float32)
X_val_tensor = torch.tensor(X_val, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_mlp, dtype = torch.float32)

y_tr_tensor = torch.tensor(
    y_tr.values,
    dtype=torch.float32
).reshape(-1, 1) #column 차원 1로 맞춰줌

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
).reshape(-1, 1)

In [ ]:
# 먼저 X와 y를 묶어 TensorDataset으로 변환
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(
    X_tr_tensor, y_tr_tensor
)

val_dataset = TensorDataset(
    X_val_tensor, y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor, y_test_tensor
)

In [ ]:
X_tr_tensor.shape

In [ ]:
# DataLoader 생성 -> batch로 쪼개준다.
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size = 256,
    shuffle = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size = 256,
    shuffle = False
)

test_loader = DataLoader(
    test_dataset,
    batch_size = 256,
    shuffle = False
)

In [ ]:
# MLP (다층 퍼셉트론: Multi-Layer-Perceptron) 모델 정의
import torch.nn as nn

class BatchNormMLP(nn.Module):
  def __init__(self, input_dim):
    super().__init__()

    self.model = nn.Sequential(
        nn.Linear(input_dim, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),

        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),

        nn.Linear(64, 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),

        nn.Linear(32, 1)
    )

  def forward(self, x):
    return self.model(x)

In [ ]:
# 모델 생성
input_dim = X_tr_tensor.shape[1]
model = BatchNormMLP(input_dim)

In [ ]:
# Loss, Optimizer 정의

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001
)

In [ ]:
# EarlyStopper 정의
class EarlyStopper:
    def __init__(self, patience=5, path='best_model.pth'):
        self.patience = patience
        self.path = path

        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss, model):
        # validation loss가 개선된 경우
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0

            # 현재까지 가장 좋은 모델 저장
            torch.save(model.state_dict(), self.path)

        # validation loss가 개선되지 않은 경우
        else:
            self.counter += 1
            print(
                f"EarlyStopping counter: {self.counter} / {self.patience}"
            )

            if self.counter >= self.patience:
                self.early_stop = True


In [ ]:
# EarlyStopper 객체 생성
early_stopper = EarlyStopper(
    patience = 5,
    path = 'best_batchnorm_mlp.pth'
)

In [ ]:
# 학습
epochs = 100

train_losses = []
val_losses = []

for epoch in range(epochs):

    # ================= Train =================
    model.train()

    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)


    # ================= Validation =================
    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

    val_loss /= len(val_loader)


    # 학습 곡선용 기록
    train_losses.append(train_loss)
    val_losses.append(val_loss)


    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )


    # Early Stopping
    early_stopper(val_loss, model)

    if early_stopper.early_stop:
        print("Early Stopping!")
        break

### Best model 불러오기

`best_batchnorm_mlp.pth` 파일은 미리 만들어두는 파일이 아니라, 위 학습 루프에서 `EarlyStopper`가 validation loss가 개선될 때 자동으로 저장합니다. 따라서 학습 셀을 먼저 실행한 뒤 아래 셀을 실행해야 합니다.


In [ ]:
# 학습이 끝난 뒤 가장 성능이 좋았던 모델 불러오기
import os

model_path = "best_batchnorm_mlp.pth"

if os.path.exists(model_path):
    model.load_state_dict(
        torch.load(
            model_path,
            weights_only=True
        )
    )
    print("Best model loaded!")
else:
    print("저장된 모델 파일이 없습니다. 먼저 위의 학습 셀을 실행해주세요.")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("BatchNorm MLP Learning Curve")

plt.legend()
plt.grid()

plt.show()

In [ ]:
# Test Data 예측
model.eval()

y_true = []
y_prob = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)

        # BCEWithLogitsLoss를 사용했으므로 sigmoid로 확률 변환
        probabilities = torch.sigmoid(outputs)

        y_true.extend(y_batch.cpu().numpy().flatten())
        y_prob.extend(probabilities.cpu().numpy().flatten())


## 8. Test Evaluation

최종 저장된 **BatchNorm MLP + EarlyStopping** 모델을 Test set으로 평가합니다.

- **AUC**: 예측 확률의 순위 판별 성능
- **Recall**: 실제 이탈 고객 중 모델이 이탈로 찾아낸 비율
- **F1**: Precision과 Recall의 조화평균


In [ ]:
# 확률 -> 0/1 예측값 변환
import numpy as np

y_true = np.array(y_true)
y_prob = np.array(y_prob)

# 기본 threshold = 0.5
y_pred = (y_prob >= 0.5).astype(int)


In [ ]:
# AUC / Recall / F1 계산
from sklearn.metrics import roc_auc_score, recall_score, f1_score

auc = roc_auc_score(y_true, y_prob)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"AUC    : {auc:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1     : {f1:.4f}")


In [ ]:
# 팀 모델 비교용 결과 표
result = pd.DataFrame({
    "Model": ["BatchNorm MLP + EarlyStopping"],
    "AUC": [auc],
    "Recall": [recall],
    "F1": [f1]
})

display(result.round(4))


In [ ]:
# Confusion Matrix (추가 확인용)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Stay(0)", "Churn(1)"])
disp.plot()
plt.title("BatchNorm MLP - Confusion Matrix")
plt.show()


### 최종 기록 항목

이 노트북에서 팀 요구사항인 다음 항목을 확인할 수 있습니다.

1. Train / Validation **학습 곡선**
2. Test **AUC**
3. Test **Recall**
4. Test **F1**

> 다른 팀원의 Dropout 또는 BatchNorm+Dropout 구조는 추가하지 않았습니다.
